In [3]:
from typing import List
import pandas as pd
import random
import numpy as np
from tqdm import tqdm
import ipdb
import re
from datasets import load_dataset
import os
from diversity import compression_ratio, ngram_diversity_score, extract_patterns, get_pos, pos_patterns, token_patterns, self_repetition_score
import json
from collections import Counter
from random import shuffle
from transformers import AutoTokenizer
tqdm.pandas()

import matplotlib.pyplot as plt
# import mplcursors
import seaborn as sns
%matplotlib inline
sns.set(style='darkgrid', context='notebook', rc={'figure.figsize':(14,10), 'font.family': 'Times'}, font_scale=3)

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('chained_assignment',None)

# Set random seeds for reproducibility on a specific machine
random.seed(1)
np.random.seed(1)
np.random.RandomState(1)
np.set_printoptions(precision=3)

In [4]:
def is_significantly_different(
    scores_A: List,
    scores_B: List,
    alpha: float = 0.05,
    n_trial: int = 10000,
    verbose: bool = False,
) -> bool:
    """Determine if the two lists of model performance are significantly
    different from each other by conducting paired bootstrapping test.

    ! Note: `scores_A` and `scores_B` need to be paired; otherwise, the result is not meaningful.

    Args:
        scores_A (List): First list of score.
        scores_B (List): Second list of score.
        alpha (float, optional): threshold for p-value (below which to be significant). Defaults to 0.05.
        n_trial (int, optional): number of bootstrap sampling to conduct. Defaults to 10000.
        verbose (bool, optional): Whether to print some intermediate results. Defaults to False.

    Returns:
        bool: whether scores_A and scores_B are significantly different from each other.
    """
    scores_A = np.array(scores_A)
    scores_B = np.array(scores_B)
    assert len(scores_A) == len(scores_B)

    # Get the inequality direction (or null hypothesis) we want to validate
    # (by calculating the raw average difference).
    # In this context, let's just call it "the ranking".
    scores_A_mean = scores_A.mean()
    scores_B_mean = scores_B.mean()
    delta = scores_B_mean - scores_A_mean

    count = 0
    n_boostrap = len(scores_A)
    for _ in range(n_trial):
        rand_ids = np.random.choice(len(scores_A), size=n_boostrap, replace=True)
        bootstrapped_scores_A = scores_A[rand_ids]
        bootstrapped_scores_B = scores_B[rand_ids]

        # Count how many times that the bootstrapped average *follows* the ranking
        if delta > 0:
            count += bootstrapped_scores_B.mean() > bootstrapped_scores_A.mean()
        else:
            count += bootstrapped_scores_B.mean() < bootstrapped_scores_A.mean()

    # how many times that the randomness (from bootstrap) causes the ranking to be violated.
    p = 1 - count / n_trial
    # if the amount of violation is below the specified threshold,
    # then it's significant difference.
    is_sig_diff = p <= alpha

    if verbose:
        print(f"Score_A avg: {np.round(scores_A_mean, 3)}")
        print(f"Score_B avg: {np.round(scores_B_mean, 3)}")
        print(f"Delta (B - A): {np.round(delta, 3)}")
        print(f"p: {p} (threshold = {alpha})")
        if is_sig_diff:
            print("Significant")
        else:
            print("*Not* Significant")

    return is_sig_diff

In [8]:
def calc_cr_nds_sr(responses):
    cr = compression_ratio(responses, 'gzip')
    nds = ngram_diversity_score(responses, 4)
    #CR-POS
    joined_pos, tuples = get_pos(responses)
    # ngrams_pos = token_patterns(joined_pos, 5, 10)
    cr_pos = compression_ratio(joined_pos, 'gzip')
    srep = self_repetition_score(responses, verbose=False)
    return cr, cr_pos, nds, srep

def calc_diversity(df, num_shuffles=100):
    '''
    Randomly assigns personas with prompts, calculates metrics over responses for these
    pairings, then calculates mean and S.D over 10 different random pairings
    '''
    random.seed(1)
    crs = []
    ndss = []
    crs_pos = []
    sreps = []
    new_df = df.set_index(['persona_id', 'prompt_id'])
    for _ in tqdm(range(num_shuffles)):
        # Get random personas paired with every prompt
        persona_ids_shuffled = [i for i in range(100)]
        random.shuffle(persona_ids_shuffled)
        prompt_ids = [i for i in range(100)]
        pairs = list(zip(persona_ids_shuffled, prompt_ids))
        responses = new_df.loc[pairs, 'response'].values.tolist()
        
        # Calculate metrics
        cr, cr_pos, nds, srep = calc_cr_nds_sr(responses)
    
        crs.append(cr)
        ndss.append(nds)
        crs_pos.append(cr_pos)
        sreps.append(srep)
    
    return crs, ndss, crs_pos, sreps

In [9]:
# Persona cutoff
df = pd.read_csv('../output/deepseek/deepseek-cutoff-persona/DeepSeek-V3_dolly_output.tsv', sep='\t')
df['response'] = df.response.apply(lambda x: x.strip())
df = df.drop_duplicates(subset=['prompt_id', 'persona_id'])
crs, ndss, crs_pos, sreps = calc_diversity(df, 100)

100%|█████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [09:50<00:00,  5.91s/it]


In [10]:
# Coarse persona cutoff
df = pd.read_csv('../output/deepseek/deepseek-coarse-cutoff/DeepSeek-V3_dolly_output.tsv', sep='\t')
df['response'] = df.response.apply(lambda x: x.strip())
df = df.drop_duplicates(subset=['prompt_id', 'persona_id'])
crs2, ndss2, crs_pos2, sreps2 = calc_diversity(df,100)

100%|█████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [09:53<00:00,  5.93s/it]


In [15]:
is_significantly_different(crs, crs2, verbose=True)

Score_A avg: 2.203
Score_B avg: 2.237
Delta (B - A): 0.033
p: 0.0 (threshold = 0.05)
Significant


True

In [16]:
is_significantly_different(ndss, ndss2, verbose=True)

Score_A avg: 3.376
Score_B avg: 3.367
Delta (B - A): -0.009
p: 0.0 (threshold = 0.05)
Significant


True

In [14]:
is_significantly_different(crs_pos, crs_pos2, verbose=True)

Score_A avg: 4.71
Score_B avg: 4.775
Delta (B - A): 0.065
p: 0.0 (threshold = 0.05)
Significant


True